# Neutralization Curves — DMS Validations

Fits Hill curves to fraction-infectivity data using `neutcurve`, then plots interactive neutralization curves with Altair for DMS validation variants."

In [1]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install neutcurve altair vl-convert-python

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import altair as alt
import neutcurve
# import vl_convert as vlc

print(f"neutcurve  {neutcurve.__version__}")
print(f"altair     {alt.__version__}")

neutcurve  2.3.0
altair     6.0.0


## 1 · Load data and fit curves

In [17]:
# ── Paths ─────────────────────────────────────────────────────────────────────
FRAC_INFECT_CSV = './data/fracinfect.csv'   # path to your fraction-infectivity CSV
OUTPUT_html      = './neutralization_curves_dms_validations.html'

# ── Load ──────────────────────────────────────────────────────────────────────
frac_infect = pd.read_csv(FRAC_INFECT_CSV)
print(frac_infect.shape)
frac_infect.head()

(168, 5)


,serum,virus,replicate,concentration,fraction infectivity
0,tufted-duck-MHCII-gag-VLP,A137S,1,0.333333,0.036589
1,tufted-duck-MHCII-gag-VLP,A137S,1,0.066667,0.036965
2,tufted-duck-MHCII-gag-VLP,A137S,1,0.013333,0.057567
3,tufted-duck-MHCII-gag-VLP,A137S,1,0.002667,0.094299
4,tufted-duck-MHCII-gag-VLP,A137S,1,0.000533,0.152838


In [18]:
# ── Fit Hill curves ───────────────────────────────────────────────────────────
fits = neutcurve.CurveFits(frac_infect)

fitparams = (
    fits.fitParams(ics=[50])
    .assign(NT50=lambda x: 1 / x['ic50'])
)
fitparams['ic50_is_bound'] = fitparams['ic50_bound'].apply(
    lambda x: x != 'interpolated'
)

fitparams[['serum', 'virus', 'ic50', 'ic50_bound', 'NT50']]

,serum,virus,ic50,ic50_bound,NT50
0,tufted-duck-MHCII-gag-VLP,A137S,0.000008,interpolated,124951.543731
1,tufted-duck-MHCII-gag-VLP,R227G,0.000004,upper,234356.690884
2,tufted-duck-MHCII-gag-VLP,T219E,0.000004,upper,234356.690884
3,tufted-duck-MHCII-gag-VLP,Y256W,0.000051,interpolated,19542.314210
4,tufted-duck-MHCII-gag-VLP,I80L,0.000033,interpolated,30221.336028
5,tufted-duck-MHCII-gag-VLP,I80L_A137S_R227G,0.000004,upper,234356.690884
6,tufted-duck-MHCII-gag-VLP,A137S_R227,0.000004,upper,234356.690884
7,tufted-duck-MHCII-gag-VLP,D77E,0.333333,lower,3.000000
8,tufted-duck-MHCII-gag-VLP,R149T,0.333333,lower,3.000000
9,tufted-duck-MHCII-gag-VLP,WT,0.000051,interpolated,19620.815986


In [19]:
# ── Optionally export fit parameters ─────────────────────────────────────────
fitparams.to_csv('ICXX_dms_validations.csv', index=False)

## 2 · Generate smooth fit curves

In [20]:
smooth_rows = []

for (serum, virus), group in frac_infect.groupby(['serum', 'virus']):
    try:
        curve = fits.getCurve(serum=serum, virus=virus, replicate='average')
    except Exception as e:
        print(f"  ⚠  Could not get curve for {serum}/{virus}: {e}")
        continue

    concs = np.logspace(
        np.log10(group['concentration'].min()),
        np.log10(group['concentration'].max()),
        200,
    )
    fit_vals = curve.fracinfectivity(concs)

    for c, v in zip(concs, fit_vals):
        smooth_rows.append(
            {'serum': serum, 'virus': virus, 'concentration': c, 'fit': float(v)}
        )

smooth_df = pd.DataFrame(smooth_rows)
print(f"{len(smooth_df)} smooth-curve points across {smooth_df['virus'].nunique()} viruses")
smooth_df.head()

2200 smooth-curve points across 11 viruses


,serum,virus,concentration,fit
0,tufted-duck-MHCII-gag-VLP,A137S,0.000004,0.561086
1,tufted-duck-MHCII-gag-VLP,A137S,0.000005,0.555635
2,tufted-duck-MHCII-gag-VLP,A137S,0.000005,0.550170
3,tufted-duck-MHCII-gag-VLP,A137S,0.000005,0.544694
4,tufted-duck-MHCII-gag-VLP,A137S,0.000005,0.539206


## Build the Altair chart

In [21]:
# ── Virus order and color mapping ─────────────────────────────────────────────
virus_order = [
    'WT', 'D77E', 'I80L', 'A137S', 'R149T',
    'R227G', 'T219E', 'Y256W', 'A137S_R227', 'I80L_A137S_R227G', 'I80L_A137S_R227G_T219E',
]

tableau10 = [
    '#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f',
    '#edc948', '#b07aa1', '#ff9da7', '#9c755f', '#bab0ac','#7EB77F'
]

color_map = {}
t10_idx = 0
for v in virus_order:
    if v == 'WT':
        color_map[v] = '#000000'
    else:
        color_map[v] = tableau10[t10_idx % len(tableau10)]
        t10_idx += 1

COLOR_SCALE = alt.Scale(domain=list(color_map.keys()), range=list(color_map.values()))  # ← NOT overwritten below

# ── Aggregate data ────────────────────────────────────────────────────────────
frac_infect_agg = (
    frac_infect
    .groupby(['serum', 'virus', 'concentration'])
    .agg({'fraction infectivity': ['mean', 'std', 'count']})
    .reset_index()
)
frac_infect_agg.columns = ['serum', 'virus', 'concentration', 'mean_frac_infectivity', 'std_frac_infectivity', 'n_replicates']
frac_infect_agg['std_frac_infectivity'] = frac_infect_agg['std_frac_infectivity'].fillna(0)
frac_infect_agg['upper_bound'] = np.clip(frac_infect_agg['mean_frac_infectivity'] + frac_infect_agg['std_frac_infectivity'], 0, 1)
frac_infect_agg['lower_bound'] = np.clip(frac_infect_agg['mean_frac_infectivity'] - frac_infect_agg['std_frac_infectivity'], 0, 1)

# ── Shared encoding helpers ───────────────────────────────────────────────────
x_enc = alt.X(
    'concentration:Q',
    scale=alt.Scale(type='log'),
    title='VLP dilution',
    axis=alt.Axis(format='.0e', labelAngle=-45, grid=False),
)
y_enc = alt.Y(
    'fit:Q',
    title='fraction infectivity',
    scale=alt.Scale(domain=[0, 1.1]),
    axis=alt.Axis(grid=False),
)
color_enc = alt.Color(          # ← used in ALL layers
    'virus:N',
    scale=COLOR_SCALE,
    title='Virus',
    sort=virus_order,
)

# ── Layer 1: smooth fitted lines ──────────────────────────────────────────────
line_layer = (
    alt.Chart(smooth_df)
    .mark_line(strokeWidth=3)
    .encode(
        x=x_enc,
        y=y_enc,
        color=color_enc,
        tooltip=[
            alt.Tooltip('virus:N', title='Virus'),
            alt.Tooltip('concentration:Q', title='Concentration', format='.4e'),
            alt.Tooltip('fit:Q', title='Fitted frac. infect.', format='.3f'),
        ],
    )
)

# ── Layer 2: error bars ───────────────────────────────────────────────────────
error_layer = (
    alt.Chart(frac_infect_agg)
    .mark_errorbar(opacity=0.6, thickness=2, size=8)
    .encode(
        x=alt.X('concentration:Q', scale=alt.Scale(type='log')),
        y=alt.Y('lower_bound:Q', title='fraction infectivity'),
        y2='upper_bound:Q',
        color=color_enc,        # ← was inline alt.Color, now shared
    )
)

# ── Layer 3: mean data points ─────────────────────────────────────────────────
scatter_layer = (
    alt.Chart(frac_infect_agg)
    .mark_point(size=130, filled=True, opacity=0.85, stroke='white', strokeWidth=1.5)
    .encode(
        x=alt.X('concentration:Q', scale=alt.Scale(type='log')),
        y=alt.Y('mean_frac_infectivity:Q', title='fraction infectivity'),
        color=color_enc,        # ← was inline alt.Color, now shared
        tooltip=[
            alt.Tooltip('virus:N', title='Virus'),
            alt.Tooltip('serum:N', title='Serum'),
            alt.Tooltip('concentration:Q', title='Concentration', format='.4e'),
            alt.Tooltip('mean_frac_infectivity:Q', title='Mean frac. infectivity', format='.3f'),
            alt.Tooltip('std_frac_infectivity:Q', title='Std dev', format='.3f'),
            alt.Tooltip('n_replicates:Q', title='N replicates'),
        ],
    )
)

# ── Combine layers ────────────────────────────────────────────────────────────
chart = (
    (line_layer + error_layer + scatter_layer)
    .properties(
        width=300,
        height=300,
        title=alt.TitleParams(
            'tufted-duck-MHCII-gag-VLP neutralization — DMS validations',
            fontSize=17,
            anchor='start',
        ),
    )
    .resolve_scale(color='shared', shape='shared')
    .configure_axis(labelFontSize=12, titleFontSize=13)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)
chart

alt.LayerChart(...)

In [23]:
# ── Save chart as html ─────────────────────────────────────────────────────────
# save as HTML
chart.save(OUTPUT_html)
print(f"✓ Chart saved as HTML: {OUTPUT_html}")

✓ Chart saved as HTML: ./neutralization_curves_dms_validations.html
